---
## Hausaufgabe 3
---

### Achtung: Überprüfen Sie vor Abgabe der Hausaufgabe, ob das Notebook richtig gespeichert wurde. 
Die Speicherung von Notebooks funktioniert über das "Disketten"-Symbol im Notebook oder über die Shortcuts Strg + S (Win) und CMD + S (MAC).

---
## Bewertung
---

#### Erreichbare Punkte: 25
---
#### Erreichte Punkte:   -
---
---
Aufgabe 1:<br>
<br>
Aufgabe 2:<br>

---

#### Kontext - Das Job Shop Problem

Neben dem Flow Shop Problem ist das Job Shop Problem eines der am weitesten verbreiteten Problemstellungen im Bereich der Maschinenbelegungsplanung. Im Unterschied zum Flow Shop existieren beim Job Shop unterschiedliche Bearbeitungsreihenfolgen für jeden Auftrag. Bspw. könnte Auftrag 1 die Bearbeitungsreihenfolge M1 - M3 - M2 haben, während Auftrag 2 Reihenfolge M3 - M1 - M2 benötigt.

Ausgehend von dieser Beschreibung beschäftigt sich die nachfolgende Hausaufgabe explizit mit dem **JOB SHOP PPROBLEM**.

---

#### Aufgabe 1 - Datenimport und Bewertung (12 Punkte)

##### a.) (2 Punkte)

Machen Sie sich mit den Dateien InputData.py, OutputData.py, JSP.json, und read.me vertraut. Importieren Sie anschließend alle Klassen und Objekte aus den mit gelieferten .py-Dateien. Erzeugen Sie anschließend das Objekt **data** als Instanz der Klasse *InputData*. Als Inputdaten dienen Ihnen die Daten aus der mitgelieferten .json-Datei (JSP.json). 


Lassen Sie sich anschließend jeden InputJob mit den Operationen, Zeiten und den zugehörigen Maschinen anzeigen.


In [ ]:
# Bitte schreiben Sie hier Ihren Code

##### b.) (2 Punkte)

Objekte der Klasse *Solution* benötigen bei Intialisierung einen Parameter *permutation*. Dieser stellt eine Sequenz mit Wiederholung dar: Eine JobId wird in dieser Sequenz entsprechend Ihrer Operations-/ Maschinenanzahl wiederholt. Für die Beispieldatei *JSP.json* ergibt sich somit eine Sequenz aus neun Ziffern (3x3).

**Beispiel**

<details>
    - Es sind drei Aufträge (Jobs) und zwei Maschinen gegeben.<br>
    - Eine gültige Permutation ergibt sich somit aus 6 Ziffern: bspw. [1, 2, 3, 1, 2, 3]<br>
    - Die erste "1" stellt in der Permutation die erste Operation des Auftrags 1 dar, die zweite "1" die zweite Operation, usw.<br>
    - In dem genannten Beispiel würde als erstes die erste Operation von Auftrag 1, dann die erste Operation von Auftrag 2 eingeplant werden.
</details>

Erstellen Sie ein Objekt **simpleSolution** der Klasse *Solution*, bei dem die Aufträge für alle Maschinen auf jeden Fall die Reihenfolge 1 - 2 - 3 gewählt wird.

In [ ]:
# Bitte schreiben Sie hier Ihren Code

##### c.) (8 Punkte)

Die Bewertungslogik für die Ermittlung der Makespan bei Flow Shop Problemen kennen Sie bereits aus dem Seminar. Der dazugehörige Code ist unten nochmals aufgeführt. Formulieren Sie diesen Code für den Fall des Job Shop Problems um und bestimmen Sie die Makespan von **simpleSolution**.

In [ ]:
import numpy 

class EvaluationLogic:    
    def DefineStartEnd(self, currentSolution):    
        #####
        # schedule first job: starts when finished at previous stage
        firstJob = currentSolution.OutputJobs[currentSolution.Permutation[0]]
        firstJob.EndTimes = numpy.cumsum([firstJob.ProcessingTime(x) for x in range(len(firstJob.EndTimes))])
        firstJob.StartTimes[1:] = firstJob.EndTimes[:-1]
        #####
        # schedule further jobs: starts when finished at previous stage and the predecessor is no longer on the considered machine
        for j in range(1,len(currentSolution.Permutation)):
            currentJob = currentSolution.OutputJobs[currentSolution.Permutation[j]]
            previousJob = currentSolution.OutputJobs[currentSolution.Permutation[j-1]]
            # first machine
            currentJob.StartTimes[0] = previousJob.EndTimes[0]
            currentJob.EndTimes[0] = currentJob.StartTimes[0] + currentJob.ProcessingTime(0)
            # other machines
            for i in range(1,len(currentJob.StartTimes)):
                currentJob.StartTimes[i] = max(previousJob.EndTimes[i], currentJob.EndTimes[i-1])
                currentJob.EndTimes[i] = currentJob.StartTimes[i] + currentJob.ProcessingTime(i)
        #####
        # Save Makespan and return Solution
        currentSolution.Makespan = currentSolution.OutputJobs[currentSolution.Permutation[-1]].EndTimes[-1]

In [ ]:
# Bitte schreiben Sie hier Ihren Code

Erwarteter Output:

    The permutation [1, 1, 1, 2, 2, 2, 3, 3, 3] results in a Makespan of 52

#### Aufgabe 2 - Das Shifting-Bottleneck-Verfahren (13 Punkte)
Das Shifting-Bottleneck-Verfahren zählt zu den besten Eröffnungsverfahren für das Job Shop Problem. Nach der erstmaligen Entwicklung durch Adams, Balas und Zawack im Jahr 1988, wurde das Verfahren auf eine Vielzahl von anderen Problemen angewendet und weiterentwickelt.

Die Grundidee besteht darin, dass in jeder Iteration eine bislang noch nicht eingeplante Maschine als Engpass identifiziert wird und anschließend die Reihenfolge der Aufträge auf dieser Maschine bestimmt wird.

---

##### a.) Vorbereitende Maßnahmen (4 Punkte)

Extrahieren Sie die Bearbeitungszeiten und die Maschinensequenzen für alle Aufträge aus **data** und speichern Sie diese Informationen in zwei separate Listen **processingTimes** und **machineSequences** als geschachtelte Listen oder Matrizen ab. Ermitteln Sie im Anschluss für jede Maschine einzeln die Vorlauf-, Bearbeitungs- und Nachlaufzeiten für jeden Auftrag:

**Vorlaufzeiten (V)**: Summe aller Bearbeitungszeiten, die aufgrund der Maschinensequenz vor der aktuellen Operation erfolgt sein müssen.<br>
**Bearbeitungszeiten (B)**: Bearbeitungszeit der aktuellen Operation.<br>
**Nachlaufzeiten (N)**: Summe aller Bearbeitungszeiten, die aufgrund der Maschinensequenz nach der aktuellen Operation erfolgen.<br>

Speichern Sie Ihre Ergebnisse als Dictionary und nutzen Sie den Maschinenindex als übergeordneten *key* und als *value* ein weiteres Dictionary mit den *keys* **V, B, N** für die Zeiten.

In [ ]:
# Bitte schreiben Sie hier Ihren Code

##### b.) Das Verfahren von Schrage (9 Punkte)

Das Verfahren von Schrage ist dient der heuristischen Minimierung der Zyklusdauer bei Vor- und Nachlaufzeiten im **Ein-Maschinen-Fall**. Das Ende der Zyklusdauer ist erreicht, wenn die Nachlaufzeit aller Aufträge abgelaufen ist. In unserem Beispiel wollen wir mit diesem Verfahren die Engpass-Maschine ermitteln.

Vorgehensweise des Verfahrens:<br>

1. Die Maschinenbelegung beginnt am Anfang des Planungszeitraums mit anschließender sukzessiver Auftragsauswahl.
2. Ein noch nicht eingeplanter Auftrag kann entweder einplanbar oder aufgrund seiner Vorlaufzeit noch nicht einplanbar sein.
3. Zu jedem Zeitpunkt, zu dem die Maschine frei ist, wird aus der Menge der gegenwärtig einplanbaren Aufträge stets der Auftrag eingeplant, dessen Nachlaufzeit am größten ist.

Ziel: Aufträge mit langer Nachlaufzeit sollen möglichst frühzeitig eingeplant werden.

Schreiben Sie eine Funktion **getBottleneck()**, die das Dictionary aus Aufgabe a als Argument erhält und anschließend für jede Maschine das Verfahren von Schrage ausführt. Im Anschluss soll die Funktion die aktuelle Engpass-Maschine zurückgeben. Testen Sie die entwickelte Funktion mit den Beispieldaten.

**Tipp:**

<details>
- Ein Beispiel des Verfahrens finden Sie in der OPM-Vorlesung Kapitel 6 (Beigefügter Ausschnitt) <br>
- Detaillierte Beschreibungen zum Verfahren finden Sie auch in Küpper, H.-U./Helber, S.: Ablauforganisation in Produktion und Logistik, 3. Aufl., Stuttgart 2004, S. 219f.
</details>

In [ ]:
# Bitte schreiben Sie hier Ihren Code